In [ ]:
%matplotlib widget
# Boilerplate import code for all libraries
# Changes to the precision require re-loading the kernel and need to be done before any op uses them.
import warpSPHCore_config as swc
from typing import Any
swc.configure(precision="float64", dim=Any) # precision: float16|half|float32|single|float64|double

import warpSPHCore as sph
from warpSPHCore.type_config import *
print(get_type_config()) # confirms active settings

# Initialize warp at this point
import warp as wp; wp.init()

import os
import torch
if torch.cuda.is_available(): # set the TORCH_CUDA_ARCH_LIST environment variable to the compute capability of the GPU for faster compiles
    os.environ['TORCH_CUDA_ARCH_LIST'] = f'{torch.cuda.get_device_properties(0).major}.{torch.cuda.get_device_properties(0).minor}'

import warnings
from tqdm import TqdmExperimentalWarning
warnings.filterwarnings("ignore", category=TqdmExperimentalWarning)
from tqdm.autonotebook import tqdm

# final import blocks that are generic
import matplotlib.pyplot as plt
from torch.profiler import profile, record_function, ProfilerActivity
import numpy as np
import math
import shlex    
import subprocess
import shutil

# custom SPH libraries
from warpSPHIntegrators.integration import *
from warpSPHCore import *

# This library
from warpSPH import *
from warpSPH.modules.timestep.compressible import computeTimestep

# The case utilities that contain all the case setup functions for the various test cases
from warpSPH.caseUtils import *

# Kidder Isentropic Compression

This notebook runs the Kidder Isentropic Compression benchmark in the compressible SPH suite.

The case models a smooth isentropic compression/expansion process and is commonly used to check entropy conservation, symmetry, and adiabatic consistency.

This notebook follows the same reusable structure used across all 15 compressible benchmark cases:

1. Configure imports and numeric precision.
2. Define case-specific physical parameters and initial-condition data.
3. Build domain, solver, and scheme configuration from shared builders.
4. Sample and initialize particles/state for the selected case.
5. Run the time integration loop with diagnostics and adaptive timestep control.
6. Export trajectory/state snapshots and generate image frames during the run.
7. Finalize outputs by writing final state data and rendering media artifacts (for example MP4/GIF).

Precision note: switching between single and double precision is controlled in the import/configuration block. Because precision is set when core modules/kernels are initialized, any precision change requires a kernel restart before re-running the notebook.

![](outputs/03-Kidder_Isentropic_compression.gif)

In [ ]:
nx = 100
dim = 1
band = 10

r_inner = 0.9
r_outer = 1.0

nu = dim
gamma = 1 + 2 / nu
rho0 = 1.0


P_inner = 0.1
P_outer = 1.0

rho_outer = 0.01
rho_inner = (P_inner / P_outer)** (1/gamma) * rho_outer # Note that the paper disagrees with this and uses outer/inner. SPHERAL uses the flipped version
s = P_outer / rho_outer**gamma # paper uses inner/inner but these are equivalent here as the case is isentropic

print(f'r_inner = {r_inner}, r_outer = {r_outer}')
print(f'P_inner = {P_inner}, P_outer = {P_outer}')
print(f'rho_inner = {rho_inner}, rho_outer = {rho_outer}')
print(f's = {s}')

extraData = {
    'nx': nx,
    'dim': dim,
    'band': band,
    'r_inner': r_inner,
    'r_outer': r_outer,
    'P_inner': P_inner,
    'P_outer': P_outer,
    'rho_inner': rho_inner,
    'rho_outer': rho_outer,
    'rho0': rho0,
    's': s,
    'nu': nu,
    'gamma': gamma
}

In [ ]:

# rightState = sodInitialState(1, 1, 0)

L = 2
dim = 1
# n_h = 4
device = torch.device('cuda:0') if torch.cuda.is_available() else torch.device('cpu')
dtype = get_torch_precision()

config, integrator = buildConfig(
    domain = buildDomainDescription(L, dim, True, device, dtype),
    dim = dim,
    kernel = KernelFunctions.B7,
    targetNeighbors = n_h_to_nH(4, dim),
    supportMode = SupportScheme.KernelMeanSymmetric,
    gradientMode = GradientScheme.Difference,
    laplacianMode = LaplacianScheme.Brookshaw,
    integrationScheme = IntegrationSchemeType.rungeKutta2,
    samplingScheme = SamplingScheme.regular,
    device = device,
    dtype = dtype,
    dt = 1e-3,
    adaptiveDt = True,
    cflFactor=0.3,
)
config.nx = nx

config.minDt = 1e-8
# config.dx = L / (nx * 2)

scheme = CompressibleSPHScheme.CRKSPH
bundle = buildScheme(scheme)
SimulationSystem, SimulationState = bundle.SimulationSystem, bundle.SimulationState
SimulationUpdate = bundle.SimulationUpdate
fn, export_fn, import_fn = bundle.stepFunction, bundle.exportFunction, bundle.importFunction


schemeConfig = bundle.SimulationConfig()
schemeConfig.gamma = gamma
schemeConfig.rho0 = rho0


schemeConfig.viscositySwitchParams.scheme = ViscositySwitch.NoneSwitch
schemeConfig.adaptiveSupportScheme = AdaptiveSupportScheme.Owen
schemeConfig.adaptiveSupportCorrections = False

In [ ]:

compressibleSystem, kidderSolution = buildKidder(config, schemeConfig, SimulationState, SimulationSystem, r_inner, r_outer, P_inner, P_outer, rho_outer, nu, gamma)

In [ ]:
timeLimit = 0.99 * kidderSolution.tau
# dt = computeDT(particleSystem.systemState, 0.05, solverConfig) / 10
timesteps = int(timeLimit / config.dt)
# simulationState = copy.deepcopy(particleSystem)
print(f"Running with dt: {config.dt}, which gives nSteps: {timesteps}")

In [ ]:
kidderBC = buildKidderBCs(schemeConfig, kidderSolution, band)
schemeConfig.boundaryConditions.clear()
schemeConfig.boundaryConditions.append(kidderBC)

In [ ]:
runningState = compressibleSystem.initializeNewState()

kineticEnergy = 0.5 * (torch.linalg.norm(runningState.state.velocities, dim = -1) **2 * runningState.state.masses).sum()
thermalEnergy = (runningState.state.internalEnergies * runningState.state.masses).sum()
totalEnergy = kineticEnergy + thermalEnergy

In [ ]:
caseName = '03-kidderIsentropicCompression'
exportPath = prepExport(f'{caseName}', config, schemeConfig, scheme, export_fn)
exportSimulationSystem(exportPath, 'initialState', scheme, compressibleSystem, exportAdjacency = False, stages = None, exportStagesAdjacency = False, extraData = dict({
    'kineticEnergy': kineticEnergy,
    'thermalEnergy': thermalEnergy,
    'totalEnergy': totalEnergy,
    'frame_num': 0,
}, **extraData))


In [ ]:
def setupPlot(fig, axis, t):
    # print(t)
    rInner = kidderSolution.rInner(t)
    rOuter = kidderSolution.rOuter(t)

    x = np.linspace(rInner, rOuter, 1000)
    rho = kidderSolution.rho(t, x)
    P = kidderSolution.P(t, x)
    # v = kidderSolution.vr(t+660*dt.cpu().item(), x)
    v = kidderSolution.vr(t, x)

    pInner = kidderSolution.Pinner(t)
    pOuter = kidderSolution.Pouter(t)

    vInner = kidderSolution.vrInner(t)
    vOuter = kidderSolution.vrOuter(t)

    axis[0,0].plot(x, rho, 'black', ls = ':', label = 'Analytic')
    axis[0,1].axhline(pInner, color = 'black', ls = ':', alpha = 0.5)
    axis[0,1].axhline(pOuter, color = 'black', ls = ':', alpha = 0.5)
    axis[0,1].plot(x, P, 'black', ls = ':', label = 'Analytic')
    axis[0,2].plot(x, v, 'black', ls = ':', label = 'Analytic')
    axis[0,2].axhline(vInner, color = 'black', ls = ':', alpha = 0.5)
    axis[0,2].axhline(vOuter, color = 'black', ls = ':', alpha = 0.5)

    axis[0,0].set_title('Density')
    axis[0,1].set_title('Pressure')
    axis[0,2].set_title('Velocity')
    
    for ax in axis[0]:
        ax.axvline(rInner, color = 'r', ls = '--', alpha = 0.5)
        ax.axvline(rOuter, color = 'r', ls = '--', alpha = 0.5)
        ax.set_xlim(rInner * 0.99, rOuter * 1.01)

In [ ]:
fig, axis = plt.subplots(1, 3, figsize = (10, 5), squeeze=False)
setupPlot(fig, axis, (runningState.t).item() if isinstance(runningState.t, torch.Tensor) else runningState.t)
axis[0,0].scatter(runningState.state.positions.cpu().numpy(), runningState.state.densities.cpu().numpy(), s = 1)
axis[0,1].scatter(runningState.state.positions.cpu().numpy(), runningState.state.pressures.cpu().numpy(), s = 1)
axis[0,2].scatter(runningState.state.positions.cpu().numpy(), runningState.state.velocities.cpu().numpy(), s = 1)
fig.suptitle(f'Kidder Isentropic Compression, t = {runningState.t:2f}, dt = {config.dt:.3g}, ptcls = {len(runningState.state.positions)}\nTotal Energy: {totalEnergy:.3g}, Kinetic Energy: {kineticEnergy:.3g}, Thermal Energy: {thermalEnergy:.3g}')

fig.tight_layout()


imagePath = f'{exportPath}/images'
os.makedirs(imagePath, exist_ok = True)
fig.savefig(f'{imagePath}/frame_{0:05d}.png')

In [ ]:
from warpSPH.modules.timestep.compressible import computeTimestep

runningState = compressibleSystem.initializeNewState()
config.dt = computeTimestep(runningState, config, schemeConfig, dt = config.dt)
print(f"Initial timestep after initialization: {config.dt}")

trajectory = []

priorStep = None
i = 0
t = 0
tq = tqdm(total = 1000, leave = True)

while t < timeLimit:
    begin = torch.cuda.Event(enable_timing=True)
    end = torch.cuda.Event(enable_timing=True)
    begin.record()
    result = integrator.function(
        state = runningState,
        f = fn,
        dt = config.dt,  
        config = config,
        schemeConfig = schemeConfig,
        verbose = False,
        # dsphConfig = solverConfig
        # priorStep = priorStep
    )
    t = result.state.t.item() if isinstance(result.state.t, torch.Tensor) else result.state.t
    tq.n = int(t / timeLimit * 1000)

    v = kidderSolution.vr(result.state.t.cpu().numpy() if isinstance(result.state.t, torch.Tensor) else result.state.t, result.state.state.positions.cpu().numpy()[:,0])
    
    result.state.state.velocities[:band,0] = torch.tensor(v[:band], dtype=result.state.state.velocities.dtype, device = result.state.state.velocities.device)
    result.state.state.velocities[-band:,0] = torch.tensor(v[-band:], dtype=result.state.state.velocities.dtype, device = result.state.state.velocities.device)
    # result.state.state.velocities[:,0] = torch.tensor(v, dtype=result.state.state.velocities.dtype, device = result.state.state.velocities.device)
    # result.state.state.internalEnergies = torch.tensor(kidderInternalEnergy(result.state.t, result.state.state.positions), dtype=result.state.state.internalEnergies.dtype, device = result.state.state.internalEnergies.device).flatten()

    end.record()
    torch.cuda.synchronize()
    priorStep = result.stages[-1]
    timing = begin.elapsed_time(end)

    # print(f"Step {i}: Time = {t:.4g}, Step Time = {timing:.3f} ms, vel nans: {torch.isnan(result.state.state.velocities).sum()}, infs: {torch.isinf(result.state.state.velocities).sum()}, pos nans: {torch.isnan(result.state.state.positions).sum()}, infs: {torch.isinf(result.state.state.positions).sum()}")

    runningState = result.state
    kineticEnergy = 0.5 * (torch.linalg.norm(runningState.state.velocities, dim = -1) **2 * runningState.state.masses).sum()
    thermalEnergy = (runningState.state.internalEnergies * runningState.state.masses).sum()
    totalEnergy = kineticEnergy + thermalEnergy

    trajectory.append(
        (i, (i+1)*config.dt, totalEnergy.item(), kineticEnergy.item(), thermalEnergy.item(), timing)
,     )

    config.dt = computeTimestep(runningState, config, schemeConfig, dt = config.dt)
    # print(f"Computed new timestep: {config.dt}")

    i = i+1

    tq.set_description(f"{i:6d}: time: {t:8.4g}/{timeLimit:8.4g}, TE: {totalEnergy:.3g}, KE: {kineticEnergy:.3g}, IE: {thermalEnergy:.3g}, step time: {timing:.3f}ms, adaptive dt: {config.dt:.3g}")
    # t = {runningState.t:2f}, dt = {config.dt:.3g}, ptcls = {len(runningState.state.positions)}\nTotal Energy: {totalEnergy:.3g}, Kinetic Energy: {kineticEnergy:.3g}, Thermal Energy: {thermalEnergy:.3g}'
    # break
    # print(runningState.t)
    if i % 100 == 0 or i == timesteps - 1 or t >= timeLimit:
        axis[0,0].cla()
        axis[0,1].cla()
        axis[0,2].cla()

        setupPlot(fig, axis, runningState.t.item() if isinstance(runningState.t, torch.Tensor) else runningState.t)
        # plotState(fig, axis, runningState)
        # fig.tight_layout()
        axis[0,0].scatter(runningState.state.positions.cpu().numpy(), runningState.state.densities.cpu().numpy(), s = 1)
        axis[0,1].scatter(runningState.state.positions.cpu().numpy(), runningState.state.pressures.cpu().numpy(), s = 1)
        axis[0,2].scatter(runningState.state.positions.cpu().numpy(), runningState.state.velocities.cpu().numpy(), s = 1)
        fig.suptitle(f'Kidder Isentropic Compression, t = {runningState.t:2e}, dt = {config.dt:.3g}, ptcls = {len(runningState.state.positions)}\nTotal Energy: {totalEnergy:.3g}, Kinetic Energy: {kineticEnergy:.3g}, Thermal Energy: {thermalEnergy:.3g}')

        fig.canvas.draw()
        fig.canvas.flush_events()
        # break
        fig.savefig(f'{imagePath}/frame_{i:05d}.png')
    if i % 500 == 0:
        exportSimulationSystem(exportPath, f'state_{i:04d}', scheme, runningState, exportAdjacency = False, stages = result.stages, exportStagesAdjacency = True, extraData = dict(**extraData, **{
            'kineticEnergy': kineticEnergy,
            'thermalEnergy': thermalEnergy,
            'totalEnergy': totalEnergy,
            'frame_num': i,
        }))
        

In [ ]:
exportSimulationSystem(exportPath, f'finalState', scheme, runningState, exportAdjacency = False, stages = result.stages, exportStagesAdjacency = True, extraData = dict(**extraData, **{
    'kineticEnergy': kineticEnergy,
    'thermalEnergy': thermalEnergy,
    'totalEnergy': totalEnergy,
    'frame_num': i,
}))

In [ ]:
ffmpeg_cmd = "ffmpeg -y -loglevel error -hide_banner -framerate 50 -f image2 -pattern_type glob -i 'frame_*.png' -c:v libx264 -pix_fmt yuv420p -b:v 10M output.mp4"
subprocess.run(shlex.split(ffmpeg_cmd), check=True, cwd = imagePath)
ffmpeg_cmd = 'ffmpeg -y -loglevel error -hide_banner -i output.mp4  -vf "fps=50,scale=540:-1:flags=lanczos,palettegen" palette.png'
subprocess.run(shlex.split(ffmpeg_cmd), check=True, cwd = imagePath)
ffmpeg_cmd = 'ffmpeg -y -loglevel error -hide_banner -i output.mp4 -i palette.png -filter_complex "fps=25,scale=540:-1:flags=lanczos[x];[x][1:v]paletteuse" out.gif'
subprocess.run(shlex.split(ffmpeg_cmd), check=True, cwd = imagePath)

# now copy the output.mp4 and out.gif to the parent directory for easier access
shutil.copy(f'{imagePath}/output.mp4', f'{exportPath}/output.mp4')
shutil.copy(f'{imagePath}/out.gif', f'{exportPath}/out.gif');